# 최종 모델 학습 — seed 17 / EfficientNet-B0 4모델 / T0.3 TTA

baseline_기본.ipynb의 흐름에 맞춰 실제 학습 코드에서 비교·분석·패키지 검증 부분을 제거한 제출용 정리본입니다.

- 분할 seed 17, 모델 seed 17·18·19·20.
- 정사각형 / 직사각형 허용 / 정사각형+가림 / 정사각형+원근.
- 학습 입력 192×192, 검증·추론은 192×192 중앙 크롭.
- 최저 검증 손실 체크포인트로 네 모델을 결합하고 원본·좌우 반전 TTA에 조건부 T0.3을 적용합니다.
- 학습에 필요한 검증 손실·학습률 전환·체크포인트 선택은 그대로 유지합니다.

이미 제공된 model.pt로 추론할 때는 이 노트북을 실행하지 않고 README.txt의 predict.py 명령만 사용합니다.
training_record.json에 기록된 노트북 해시는 실제 학습 당시 원본의 해시이며, 이 정리본은 출력과 보조 셀 제거로 파일 해시가 다릅니다.


## 실행 준비

학습 재현 시 별도 작업 폴더에 이 노트북과 train/6e, train/it, test, sample_submission.csv를 준비하고 위에서부터 실행합니다.
학습 결과와 새 예측은 실험결과/011_192크롭_패딩제거/seed_17/에 저장합니다. 현재 최종 폴더의 model.pt·submission.csv를 덮어쓰지 않습니다.
추론용 model.py·predict.py도 아래 셀에서 생성합니다. 학습 중 ImageNet 사전학습 가중치를 다운로드할 수 있으며 추론 중에는 다운로드하지 않습니다.


## 모듈·seed


In [ ]:
import csv
import importlib
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 17
BASE_DIR = Path.cwd().resolve()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(SEED)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
	print(torch.cuda.get_device_name())


## 학습 설정


In [ ]:
params = {
	'epoch': 50,
	'early_stopping': True,
	'min_epochs': 5,
	'patience': 5,
	'batch_size': 32,
	'learning_rates': (0.001, 0.0003, 0.0001),
	'valid_ratio': 0.15,
}


In [ ]:
RUN_SEEDS = tuple(range(SEED, SEED + 4))
AUGMENTATION_BY_SEED = dict(zip(RUN_SEEDS, ('square', 'rectangle', 'square_erasing', 'square_perspective')))
RESULT_DIR = BASE_DIR / '실험결과' / '011_192크롭_패딩제거' / f'seed_{SEED}'
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
EPOCH_DIR = RESULT_DIR / 'epochs'
assert not (RESULT_DIR / 'runs.csv').exists(), f'기존 학습 결과를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not (RESULT_DIR / 'model.pt').exists(), f'기존 가중치를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not list(CHECKPOINT_DIR.glob('*.pt')), f'체크포인트가 이미 있습니다: {CHECKPOINT_DIR}'
for directory in (CHECKPOINT_DIR, EPOCH_DIR):
	directory.mkdir(parents=True, exist_ok=True)
print('결과 위치:', RESULT_DIR)


## 모델·전처리 정의


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as F

IMG_SIZE = 192
MIN_CROP_SIZE = 96
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PERSPECTIVE_FILL = tuple(round(value * 255) for value in NORM_MEAN)
TEMPERATURE_RULES = ("T0.3", "T0.5")


class RandomPhotoCrop:
	def __init__(self, square):
		self.square = square

	def __call__(self, image):
		width, height = image.size
		assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작은 원본입니다: {image.size}"
		crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		if self.square:
			crop_height = crop_width
		else:
			crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
			while (crop_width / crop_height < 0.75) or (crop_width / crop_height > 4 / 3):
				crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
				crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		top = int(torch.randint(height - crop_height + 1, ()).item())
		left = int(torch.randint(width - crop_width + 1, ()).item())
		return F.crop(image, top, left, crop_height, crop_width)


def center_crop(image):
	width, height = image.size
	assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192より小さいためパディングなしで切り出せません: {image.size}"
	return F.center_crop(image, [IMG_SIZE, IMG_SIZE])


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(center_crop),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs, temperature_rule="T0.3"):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 4)
	assert bool(torch.isfinite(member_probs).all()), "モデル確率に非有限値があります。"
	assert bool(((member_probs >= 0) & (member_probs <= 1)).all()), "モデル確率が範囲外です。"
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	middle_mean = member_probs.sort(dim=1).values[:, 1:3].mean(dim=1, keepdim=True)
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), middle_mean)
	if temperature_rule == "T0.3":
		temperatures = torch.where(unanimous, 0.3, 0.7)
	elif temperature_rule == "T0.5":
		temperatures = 0.5
	else:
		raise ValueError(f"알 수 없는 temperature 규칙: {temperature_rule}")
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(pretrained=False):
	weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
	net = models.efficientnet_b0(weights=weights)
	net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	return net


class EfficientNetEnsemble(nn.Module):
	def __init__(self, state_dicts):
		super().__init__()
		assert len(state_dicts) == 4, "앙상블 가중치 4개가 필요합니다."
		self.models = nn.ModuleList()
		for state_dict in state_dicts:
			model = build_model()
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs):
		return torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)

	def forward(self, inputs, temperature_rule="T0.3"):
		return combine_probabilities(self.member_probabilities(inputs), temperature_rule)


In [ ]:
import model as model_module
importlib.invalidate_caches()
importlib.reload(model_module)
assert Path(model_module.__file__).resolve() == BASE_DIR / 'model.py'
model = model_module.build_model()
print('EfficientNet-B0 / input:', model_module.IMG_SIZE)


### 전체 파라미터 수


In [ ]:
PARAM_LIMIT = 20_000_000
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f'1모델 {n_params:,} / 4모델 {4 * n_params:,} / 제한 {PARAM_LIMIT:,}')
assert 4 * n_params <= PARAM_LIMIT
del model


## 학습 데이터·증강

각 변 96~192픽셀을 원본에서 크롭한 뒤 192×192로 확대합니다.
정사각형 모델은 가로·세로 길이가 같고, 직사각형 모델은 가로/세로 0.75~1.333 범위에서 비율 변형을 허용합니다.
좌우 반전 확률 50%, seed별 색상·가림·원근 설정은 실제 학습과 동일합니다.


In [ ]:
data_path = BASE_DIR / 'train'
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD
AUGMENTATION_SETTINGS = {
	'square': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.0},
	'rectangle': {'square': False, 'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.3, 'erase_p': 0.0, 'perspective_p': 0.0},
	'square_erasing': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.1, 'perspective_p': 0.0},
	'square_perspective': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.3},
}
train_transforms = {}
for name, settings in AUGMENTATION_SETTINGS.items():
	operations = [
		model_module.RandomPhotoCrop(square=settings['square']),
		transforms.RandomHorizontalFlip(p=0.5),
		transforms.ColorJitter(brightness=settings['brightness'], contrast=settings['contrast'], saturation=settings['saturation']),
	]
	if settings['perspective_p'] > 0:
		operations.append(transforms.RandomPerspective(distortion_scale=0.15, p=settings['perspective_p'], fill=model_module.PERSPECTIVE_FILL))
	operations.extend([
		transforms.Resize((IMG_SIZE, IMG_SIZE)),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])
	if settings['erase_p'] > 0:
		operations.append(transforms.RandomErasing(p=settings['erase_p'], scale=(0.02, 0.1), value=0))
	train_transforms[name] = transforms.Compose(operations)
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transforms['square'])
full_valid = datasets.ImageFolder(data_path, valid_transform)
assert full_train.class_to_idx == {'6e': 0, 'it': 1}
assert len(full_train) == 2906, '학습' + f' 사진 수가 다릅니다: {len(full_train)}'
for path, label in full_train.samples:
	with Image.open(path) as image:
		assert min(image.size) >= IMG_SIZE, f'192보다 작은 이미지: {path}'
print('클래스:', full_train.class_to_idx, '/ 전체:', len(full_train), '/ 입력:', IMG_SIZE)


### 학습·검증 분할 — seed 17, 검증 15%


In [ ]:
labels = [label for _, label in full_train.samples]
train_idx, valid_idx = train_test_split(list(range(len(full_train))), test_size=params['valid_ratio'], stratify=labels, random_state=SEED)
assert (len(train_idx) == 2470) and (len(valid_idx) == 436)
assert not set(train_idx).intersection(valid_idx)
data_sets = {'train': Subset(full_train, train_idx), 'valid': Subset(full_valid, valid_idx)}
with (RESULT_DIR / 'validation_split.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'label'))
	writer.writerows((Path(full_valid.samples[index][0]).stem, labels[index]) for index in valid_idx)
print(f'train: {len(train_idx)} / valid: {len(valid_idx)} / split seed: {SEED}')


## 네 모델 학습

각 모델은 ImageNet 사전학습에서 시작해 전체 파라미터를 추가 학습합니다.
학습률 0.001 → 0.0003 → 0.0001. 단계별 최소 5 epoch 후 연속 5회 검증 손실 미개선 시 전환하고 마지막 단계에서는 종료합니다.
최대 50 epoch이며 전체 최저 검증 손실 가중치를 저장합니다. epoch별 학습·검증 지표는 CSV에 기록합니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
	"""optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
	training = optimizer is not None
	model.train() if training else model.eval()

	total_loss, total_correct, total_n = 0.0, 0, 0
	with torch.set_grad_enabled(training):
		for inputs, targets in loader:
			inputs = inputs.to(device)
			targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

			logits = model(inputs)
			loss = criterion(logits, targets)
			assert bool(torch.isfinite(loss)), "학습/검증 손실이 비유한 값입니다."

			if training:
				optimizer.zero_grad()
				loss.backward()
				optimizer.step()

			probs = torch.sigmoid(logits)
			preds = (probs >= 0.5).float()
			total_loss += loss.item() * inputs.size(0)
			total_correct += (preds == targets).sum().item()
			total_n += inputs.size(0)

	return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates):
	assert min_epochs > 0
	assert patience > 0
	assert len(learning_rates) > 0
	assert all(rate > 0 for rate in learning_rates)
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}', flush=True)

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
results = []
for seed in RUN_SEEDS:
	augmentation = AUGMENTATION_BY_SEED[seed]
	full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
	assert full_run_train.samples == full_train.samples
	run_data_sets = {'train': Subset(full_run_train, train_idx), 'valid': data_sets['valid']}
	print(f'\n학습 시작: seed {seed} / {augmentation}', flush=True)
	seed_everything(seed)
	model = model_module.build_model(pretrained=True).to(device)
	seed_everything(seed)
	dataloaders = {
		'train': DataLoader(run_data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed)),
		'valid': DataLoader(run_data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
	}
	criterion = nn.BCEWithLogitsLoss()
	optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rates'][0])
	pt_file_name = CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt'
	assert not pt_file_name.exists(), f'기존 가중치를 덮어쓰지 않습니다: {pt_file_name}'
	started = time.perf_counter()
	with (EPOCH_DIR / f'efficientnet_b0_seed{seed}.csv').open('w', newline='', encoding='utf-8-sig') as epoch_csv:
		history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params['epoch'], epoch_csv=epoch_csv, early_stopping=params['early_stopping'], min_epochs=params['min_epochs'], patience=params['patience'], learning_rates=params['learning_rates'])
	elapsed = time.perf_counter() - started
	results.append({
		'model': 'efficientnet_b0', 'seed': seed, 'augmentation': augmentation,
		'image_size': IMG_SIZE, 'selection_metric': 'valid_loss', 'learning_rates': str(params['learning_rates']),
		'max_epochs': params['epoch'], 'epochs_run': len(history['train_loss']),
		'early_stopping': params['early_stopping'], 'min_epochs': params['min_epochs'], 'patience': params['patience'],
		'best_epoch': best_epoch, 'valid_loss': best_loss, 'valid_acc': history['valid_acc'][best_epoch - 1],
		'seconds': elapsed, 'n_params': n_params, 'checkpoint': str(pt_file_name),
	})
	with (RESULT_DIR / 'runs.csv').open('w', newline='', encoding='utf-8-sig') as output:
		writer = csv.DictWriter(output, fieldnames=list(results[0]))
		writer.writeheader()
		writer.writerows(results)
	del model, optimizer, dataloaders
	if device.type == 'cuda':
		torch.cuda.empty_cache()
assert len(results) == 4


## 최저 손실 가중치 결합


In [ ]:
checkpoint = {
	'split_seed': SEED, 'image_size': IMG_SIZE, 'preprocessing': 'center_crop_192_no_padding',
	'seeds': list(RUN_SEEDS), 'augmentations': [AUGMENTATION_BY_SEED[seed] for seed in RUN_SEEDS],
	'state_dicts': [torch.load(CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt', map_location='cpu', weights_only=True) for seed in RUN_SEEDS],
}
torch.save(checkpoint, RESULT_DIR / 'model.pt')
ensemble = model_module.EfficientNetEnsemble(checkpoint['state_dicts']).eval()
ensemble_params = sum(parameter.numel() for parameter in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT
print('모델 저장:', RESULT_DIR / 'model.pt')


## 테스트 추론 — T0.3 TTA

검증과 동일한 중앙 크롭·정규화를 사용합니다. 각 모델의 원본·좌우 반전 확률을 평균한 뒤 네 모델을 결합합니다.
전원 일치 시 확률 평균에 T=0.3, 불일치 시 가운데 두 확률 평균에 T=0.7을 적용합니다.


In [ ]:
class TestImageDataset(Dataset):
	def __init__(self, files, transform):
		self.files = files
		self.transform = transform

	def __len__(self):
		return len(self.files)

	def __getitem__(self, index):
		path = self.files[index]
		with Image.open(path) as image:
			return self.transform(image.convert('RGB')), path.stem


test_files = sorted(path for path in (BASE_DIR / 'test').iterdir() if path.is_file() and (path.suffix.lower() in ('.jpg', '.jpeg', '.png')))
with (BASE_DIR / 'sample_submission.csv').open(newline='', encoding='utf-8-sig') as source:
	sample_rows = list(csv.DictReader(source))
expected_ids = {row['id'] for row in sample_rows}
assert (len(test_files) == 500) and (len(expected_ids) == 500)
assert len({path.stem for path in test_files}) == 500
assert {path.stem for path in test_files} == expected_ids
test_loader = DataLoader(TestImageDataset(test_files, valid_transform), batch_size=params['batch_size'], shuffle=False, num_workers=0)


In [ ]:
rows = []
with torch.inference_mode():
	for inputs, image_ids in test_loader:
		members = ensemble.member_probabilities(inputs)
		flipped_members = ensemble.member_probabilities(inputs.flip(dims=[3]))
		probs = model_module.combine_probabilities((members + flipped_members) / 2, 'T0.3').squeeze(1)
		assert bool(torch.isfinite(probs).all())
		rows.extend((image_id, f'{prob.item():.6f}') for image_id, prob in zip(image_ids, probs))
assert (len(rows) == 500) and ({row[0] for row in rows} == expected_ids)
output_path = RESULT_DIR / 'submission.csv'
with output_path.open('w', newline='', encoding='utf-8') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'prob'))
	writer.writerows(rows)
print('예측 저장:', output_path)


## CPU 추론 코드


In [ ]:
%%writefile predict.py
import argparse
import csv
from pathlib import Path

import torch
from PIL import Image

from model import IMG_SIZE, TEMPERATURE_RULES, EfficientNetEnsemble, build_valid_transform, combine_probabilities

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
	files = sorted(path for path in Path(input_dir).iterdir() if path.is_file() and (path.suffix.lower() in (".jpg", ".jpeg", ".png")))
	assert files, f"이미지가 없습니다: {input_dir}"
	assert len({path.stem for path in files}) == len(files), "이미지 ID가 중복됩니다."
	return files


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--temperature-rule", choices=TEMPERATURE_RULES, default="T0.3")
	parser.add_argument("--tta", action="store_true", default=True, help="모델별 원본·좌우 반전 확률 평균 후 결합")
	args = parser.parse_args()
	weights_path = Path(args.weights)
	if not weights_path.is_absolute():
		weights_path = Path(__file__).resolve().parent / weights_path
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE
	assert checkpoint["preprocessing"] == "center_crop_192_no_padding"
	assert len(checkpoint["seeds"]) == 4, "모델 seed 4개가 필요합니다."
	assert all(isinstance(seed, int) for seed in checkpoint["seeds"]), "모델 seed는 정수여야 합니다."
	assert checkpoint["seeds"] == list(range(checkpoint["seeds"][0], checkpoint["seeds"][0] + 4)), "연속된 모델 seed 4개가 필요합니다."
	assert checkpoint["augmentations"] == ["square", "rectangle", "square_erasing", "square_perspective"]
	model = EfficientNetEnsemble(checkpoint["state_dicts"]).eval()
	n_params = sum(parameter.numel() for parameter in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	transform = build_valid_transform()
	files = load_images(args.input)
	rows = []
	with torch.inference_mode():
		for position in range(0, len(files), 32):
			batch_files = files[position:position + 32]
			images = []
			for path in batch_files:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images)
			member_probs = model.member_probabilities(inputs)
			if args.tta:
				member_probs = (member_probs + model.member_probabilities(inputs.flip(dims=[3]))) / 2
			probs = combine_probabilities(member_probs, args.temperature_rule).squeeze(1)
			assert bool(torch.isfinite(probs).all())
			for path, prob in zip(batch_files, probs):
				rows.append((path.stem, f"{prob.item():.6f}"))
	with Path(args.output).open("w", newline="", encoding="utf-8") as output:
		writer = csv.writer(output)
		writer.writerow(("id", "prob"))
		writer.writerows(rows)
	print(f"{args.output} 저장 완료 / {len(files)}장 / {n_params:,} parameters / TTA={args.tta}")


if __name__ == "__main__":
	main()


학습을 다시 실행할 필요 없이 제공된 model.pt로 예측할 때는 이 폴더에서 다음 명령을 사용합니다.

python predict.py --input "<테스트 이미지 폴더>" --output pred.csv

재학습으로 생성된 model.pt와 submission.csv는 위 RESULT_DIR에 저장됩니다. 압축·별도 비교·자가 검증 셀은 포함하지 않습니다.
